# BlinkMimic → Ours-T4：双卡 T4 冒烟

上游：`fuyichen0725/blinkmimic`，commit `d264658`，MIT。

## 上游是怎么从 SuperMimic 改过来的

BlinkMimic 没有重写方法，而是把 SuperMimic 的每一步都做成开关，再用 `scripts/presets.sh` 的 `PRESET[main]` 组合成新配方：

| 层 | 改动 | 对应开关 |
|---|---|---|
| 训练循环 | 去掉更新循环里的 host 同步、融合 Adam/旋转、更多子图编译 | `--no_sync --fused_adam --fused_rot --compile_more` |
| 优化器 | 隐藏层权重改用 Muon | `--pi_opt muon --wm_opt muon` |
| 世界模型 | 半隐式结构、左右镜像增广、3×1024 | `--wm_structured --wm_mirror 0.5 --wm_hidden 1024,1024,1024` |
| 策略 | 参考动作残差、EMA、批 2048 + 4× 学习率衰减 | `--pi_ref_skip --pi_ema 0.995 --pi_batch 2048 --pi_lr_mult 4` |
| 采样 | 失败点前重置、评估引导重置 | `--fail_reset 0.3 --eprobe_*` |
| 系统工程 | 更新/采样/probe 用 CUDA graph | `--cuda_graph --collect_graph --eprobe_graph` |

## 我们这一版改了什么（只改 T4 必须改的）

- `--amp none`：T4 是 Turing sm_75，不支持 bf16；上游默认 `--amp bf16`。
- `num_envs 4096→512`，`pi_batch 2048→512`，`wm_batch 2048→512`：16 GB 显存。
- encoder/decoder 从 2048-1024-512-256 降到 1024-512-256。
- 首轮关闭 `--cuda_graph` / `--eprobe_*` / `--compile_more`：先验证数值、显存和吞吐，再单独开启。
- 保留 structured WM、mirror、Muon、ref skip、EMA、fused 系列、contact reward。

双卡用法：两个独立单卡进程（`CUDA_VISIBLE_DEVICES=0/1`），不是数据并行。上游 `train.py` 写死 `cuda:0`，所以不做跨卡并行改造。

本 Notebook 只验证“能不能在双 T4 上跑通并记录显存”，不产出方法结论。
> 已被 `kaggle_blinkmimic_t4_ddp.ipynb` 取代：那一版用 torchrun 双卡数据并行 + fp16。本文件保留为单卡/无 AMP 的对照记录。


In [ ]:
from pathlib import Path

REPO_DIR = Path('/kaggle/working/proj2-humanoid-mimic/third_party/blinkmimic')
REPO_URL = 'https://github.com/fuyichen0725/blinkmimic.git'
COMMIT = 'd264658d6c9ce61fb52f9f0f8985a369f3ccc409'
MOTION_SRC = Path('/kaggle/input/datasets/machineabao/smoke1')
LOG_ROOT = Path('/kaggle/working/proj2-humanoid-mimic/results/blinkmimic_t4')

NUM_ENVS = 512
PI_BATCH = 512
WM_BATCH = 512
ITERATIONS = 100
SEEDS = (0, 1)
TIMEOUT_S = 3600
LAUNCH_STAGGER_S = 20

T4_FLAGS = [
    '--num_envs', str(NUM_ENVS),
    '--iterations', str(ITERATIONS),
    '--pi_batch', str(PI_BATCH),
    '--wm_batch', str(WM_BATCH),
    '--wm_batches', '5',
    '--horizon', '32',
    '--wm_hidden', '1024,1024,1024',
    '--enc_hidden', '1024,512,256',
    '--dec_hidden', '1024,512,256',
    '--amp', 'none',
    '--lr', '4e-4',
    '--pi_lr_mult', '4',
    '--lr_decay_ret', '0.75',
    '--lr_decay_len', '300',
    '--lr_decay_by', '1400',
    '--pi_ema', '0.995',
    '--no_sync',
    '--fused_adam',
    '--fused_rot',
    '--fail_reset', '0.3',
    '--contact_reward', '0.25',
    '--wm_structured',
    '--wm_mirror', '0.5',
    '--pi_opt', 'muon',
    '--wm_opt', 'muon',
    '--pi_ref_skip',
    '--wm_only_iters', '0',
    '--bin_fix',
    '--ref_contact_fix',
    '--ckpt_light',
    '--ckpt_minutes', '0',
    '--ckpt_every', '100',
    '--log_every', '10',
    '--max_hours', '0.8',
]

LOG_ROOT.mkdir(parents=True, exist_ok=True)
print(f'REPO_DIR={REPO_DIR}')
print(f'MOTION_SRC={MOTION_SRC} exists={MOTION_SRC.exists()}')
print(f'NUM_ENVS={NUM_ENVS} PI_BATCH={PI_BATCH} WM_BATCH={WM_BATCH} ITERATIONS={ITERATIONS}')


In [ ]:
import subprocess

info = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'],
    text=True,
)
print(info.strip())
gpu_count = len([line for line in info.strip().splitlines() if line.strip()])
print(f'gpu_count={gpu_count}')
if gpu_count != 2:
    raise RuntimeError(f'需要 2 张 T4，当前 {gpu_count} 张')


In [ ]:
import subprocess

def run(command, cwd=None, check=True):
    print('$', ' '.join(str(item) for item in command))
    return subprocess.run(command, cwd=cwd, check=check, text=True)

REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO_DIR)])
run(['git', 'fetch', '--depth', '1', 'origin', COMMIT], cwd=REPO_DIR)
run(['git', 'checkout', '--detach', COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print(f'git_head={head}')
if head != COMMIT:
    raise RuntimeError('commit 校验失败')

run(['uv', 'sync', '--frozen'], cwd=REPO_DIR)

check = '''
import importlib.metadata as m, sys, torch
print(f"python={sys.version.split()[0]}")
print(f"torch={torch.__version__} cuda={torch.version.cuda}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")
for n in ("mjlab", "mujoco", "warp-lang"):
    try:
        print(f"{n}={m.version(n)}")
    except m.PackageNotFoundError:
        print(f"{n}=NOT_FOUND")
'''
run(['uv', 'run', 'python', '-c', check], cwd=REPO_DIR)


In [ ]:
import json
import shutil
import subprocess
import urllib.request
from pathlib import Path

if not MOTION_SRC.exists():
    raise FileNotFoundError(f'动作目录不存在: {MOTION_SRC}')

motion_dir = REPO_DIR / 'data' / 'lafan1_g1'
motion_dir.mkdir(parents=True, exist_ok=True)
src_csv = sorted(MOTION_SRC.glob('*.csv'))
if not src_csv:
    raise FileNotFoundError(f'{MOTION_SRC} 下没有 CSV')
for source in src_csv:
    target = motion_dir / source.name
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
print('motions:')
for item in sorted(motion_dir.glob('*.csv')):
    print(' ', item.name, item.stat().st_size // 1024, 'KB')

HF = 'https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset'
API = 'https://huggingface.co/api/datasets/lvhaidong/LAFAN1_Retargeting_Dataset/tree/main'
desc = REPO_DIR / 'data' / 'g1_description'
meshes = desc / 'meshes'
meshes.mkdir(parents=True, exist_ok=True)

def download(url, out):
    if out.exists() and out.stat().st_size > 0:
        return
    urllib.request.urlretrieve(url, out)

download(f'{HF}/resolve/main/robot_description/g1/g1_29dof_rev_1_0.urdf', desc / 'g1_29dof_rev_1_0.urdf')
with urllib.request.urlopen(f'{API}/robot_description/g1/meshes') as response:
    entries = json.load(response)
for entry in entries:
    download(f'{HF}/resolve/main/{entry["path"]}', meshes / Path(entry['path']).name)
print(f'meshes={len(list(meshes.glob("*")))}')

mjcf = desc / 'g1_29dof.xml'
if not mjcf.exists():
    subprocess.run(['uv', 'run', 'python', 'tools/urdf_to_mjcf.py'], cwd=REPO_DIR, check=True)
print(f'mjcf_exists={mjcf.exists()} size={mjcf.stat().st_size if mjcf.exists() else 0}')


In [ ]:
import subprocess

command = ['uv', 'run', 'python', 'tools/test_env.py', '--num_envs', '64', '--motion_dir', 'data/lafan1_g1']
print('$', ' '.join(command))
result = subprocess.run(command, cwd=REPO_DIR, text=True)
if result.returncode != 0:
    raise RuntimeError(f'test_env 失败 exit={result.returncode}')
print('test_env=PASS')


In [ ]:
import csv
import os
import subprocess
import threading
import time
from pathlib import Path

gpu_log = LOG_ROOT / 'gpu_samples.csv'
gpu_log.parent.mkdir(parents=True, exist_ok=True)
stop = threading.Event()

def monitor_gpu():
    with open(gpu_log, 'w', newline='') as output:
        writer = csv.writer(output)
        writer.writerow(['timestamp', 'gpu_index', 'memory_used_mib', 'utilization_pct'])
        while not stop.is_set():
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=timestamp,index,memory.used,utilization.gpu', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, check=False,
            )
            for line in result.stdout.splitlines():
                writer.writerow([part.strip() for part in line.split(',')])
            output.flush()
            stop.wait(2)

monitor = threading.Thread(target=monitor_gpu, daemon=True)
monitor.start()

processes = []
started = time.time()
for seed in SEEDS:
    log_dir = LOG_ROOT / f'seed{seed}'
    log_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(seed)
    env['OMP_NUM_THREADS'] = '2'
    env['MUJOCO_GL'] = 'egl'
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    command = ['uv', 'run', 'python', '-u', '-m', 'blinkmimic.train', '--log_dir', str(log_dir), '--seed', str(seed), *T4_FLAGS]
    print(f'$ CUDA_VISIBLE_DEVICES={seed} ' + ' '.join(command))
    handle = open(log_dir / 'stdout.log', 'w')
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=handle, stderr=subprocess.STDOUT)
    processes.append((seed, process, handle, log_dir))
    time.sleep(LAUNCH_STAGGER_S)

returncodes = {}
deadline = started + TIMEOUT_S
for seed, process, handle, log_dir in processes:
    remaining = max(1, deadline - time.time())
    try:
        returncodes[seed] = process.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        process.terminate()
        returncodes[seed] = process.wait(timeout=60)
    finally:
        handle.close()

stop.set()
monitor.join(timeout=5)
print(f'wall_seconds={time.time() - started:.1f}')

rows = list(csv.DictReader(open(gpu_log)))
for gpu_index in sorted({row['gpu_index'] for row in rows}):
    values = [int(row['memory_used_mib']) for row in rows if row['gpu_index'] == gpu_index]
    print(f'gpu_{gpu_index}_peak_memory_mib={max(values) if values else 0}')

for seed, _, _, log_dir in processes:
    lines = (log_dir / 'stdout.log').read_text(errors='replace').splitlines()
    progress = [line for line in lines if 'it ' in line]
    print(f'--- seed{seed} rc={returncodes[seed]} lines={len(lines)}')
    if progress:
        print(progress[-1])
    if returncodes[seed] != 0:
        for line in lines[-25:]:
            print(line)

if any(code != 0 for code in returncodes.values()):
    raise RuntimeError(f'训练失败: {returncodes}')
print('status=PASS_BLINKMIMIC_T4_L0')
